In [1]:
"""             Resume
                 |             
                 Extract text
                 |
        -------------------
        |        |        |
     Skills   Experience  Grammar
     Analyzer  Analyzer   Analyzer
        |        |        |
        ---------- -------
                 |
            Final Report
"""

'             Resume\n                 |             \n                 Extract text\n                 |\n        -------------------\n        |        |        |\n     Skills   Experience  Grammar\n     Analyzer  Analyzer   Analyzer\n        |        |        |\n        ---------- -------\n                 |\n            Final Report\n'

In [30]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os 

In [34]:
load_dotenv()

True

In [35]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [18]:
class skillanalyse(BaseModel):
    strengths : list[str]
    missing_skills : list[str]
    score  : int = Field(ge=0,le=10)
    

In [19]:
class experienceanalyse(BaseModel):
    year_of_exp : float
    achievements : list[str]
    score : int = Field(ge=0, le=10)
    

In [20]:
class ResumeState(TypedDict):
    resume_text :str

    skill_analysis : skillanalyse | None

    experience_analysis : experienceanalyse | None

    final_report : str | None

In [21]:
def skill_analysis(state:ResumeState)-> ResumeState:
    resume_text = state['resume_text']

    prompt = f"""Based on this resume {'resume_text'}
Analyse the skills out of it and mark the strengths and mark missing skills 
and then score it out of 10


"""
    structured_llm = llm.with_structured_output(skillanalyse)

    result = structured_llm.invoke(prompt)

    return {"skill_analysis":result}
    

In [22]:
def experience_analysis(state:ResumeState)-> ResumeState:
    resume_text = state['resume_text']

    prompt = f"""Based on this resume {resume_text}  
Analyze the candidate's Experience and tell about years of experience and achievements
and then score it out of 10  
    
    
    """
    structured_llm2 = llm.with_structured_output(experienceanalyse)

    result = structured_llm2.invoke(prompt)

    return {
        "experience_analysis": result
    }

In [23]:
def final_judgement(state:ResumeState)-> ResumeState:
    skill_analysis = state['skill_analysis']
    experience_analyis = state['experience_analysis']

    prompt = f"""Based on the candidate's skill analysis {skill_analysis} and experience analysis {experience_analysis}
Generate a final report as a final judgement about the resume 
    
    
    
    """
    result = llm.invoke(prompt).content

    return {
        'final_report':result
    }

In [27]:
graph = StateGraph(ResumeState)

graph.add_node('skill_analysis', skill_analysis)
graph.add_node('experience_analysis',experience_analysis)
graph.add_node('final_judgement',final_judgement)

graph.add_edge(START,'skill_analysis')
graph.add_edge(START,'experience_analysis')
graph.add_edge('skill_analysis','final_judgement')
graph.add_edge('experience_analysis','final_judgement')
graph.add_edge('final_judgement',END)

workflow=graph.compile()

In [28]:
resume = """Vansh Sharma
Email: vansh.sharma@gmail.com
Phone: +91-9876543210
Location: Delhi, India

SUMMARY
Computer Science undergraduate passionate about Artificial Intelligence, Machine Learning, and Backend Development. Strong problem-solving skills with experience in Python, FastAPI, and SQL. Looking for AI/ML Internship opportunities.

EDUCATION
B.Tech in Computer Science
XYZ Institute of Technology
2023 - 2027
Current CGPA: 8.6

SKILLS
Programming Languages:
- Python
- Java
- SQL

Libraries & Frameworks:
- Pandas
- NumPy
- Scikit-learn
- FastAPI
- LangChain

Tools:
- Git
- GitHub
- VS Code
- PostgreSQL

PROJECTS

1. Resume Screening System
- Built an AI-powered resume screening application using LangChain.
- Used OpenAI GPT models for candidate evaluation.
- Generated structured hiring reports.

2. Student Result Prediction
- Developed a Machine Learning model using Random Forest.
- Performed data preprocessing and feature engineering.
- Achieved 91% prediction accuracy.

EXPERIENCE

AI Intern
ABC Technologies
May 2026 - July 2026

Responsibilities:
- Developed REST APIs using FastAPI.
- Built data preprocessing pipelines.
- Assisted in training ML models.
- Worked with Git and PostgreSQL.

CERTIFICATIONS
- Machine Learning Specialization
- Python for Data Science
- SQL Intermediate

ACHIEVEMENTS
- Solved 350+ DSA problems.
- Finalist in University Hackathon.

LANGUAGES
English
Hindi
"""


In [38]:
result =workflow.invoke({'resume_text':resume})

print(result)

{'resume_text': 'Vansh Sharma\nEmail: vansh.sharma@gmail.com\nPhone: +91-9876543210\nLocation: Delhi, India\n\nSUMMARY\nComputer Science undergraduate passionate about Artificial Intelligence, Machine Learning, and Backend Development. Strong problem-solving skills with experience in Python, FastAPI, and SQL. Looking for AI/ML Internship opportunities.\n\nEDUCATION\nB.Tech in Computer Science\nXYZ Institute of Technology\n2023 - 2027\nCurrent CGPA: 8.6\n\nSKILLS\nProgramming Languages:\n- Python\n- Java\n- SQL\n\nLibraries & Frameworks:\n- Pandas\n- NumPy\n- Scikit-learn\n- FastAPI\n- LangChain\n\nTools:\n- Git\n- GitHub\n- VS Code\n- PostgreSQL\n\nPROJECTS\n\n1. Resume Screening System\n- Built an AI-powered resume screening application using LangChain.\n- Used OpenAI GPT models for candidate evaluation.\n- Generated structured hiring reports.\n\n2. Student Result Prediction\n- Developed a Machine Learning model using Random Forest.\n- Performed data preprocessing and feature engineer